# 03) Causal Variable-Length LSTM

This file starts from the ML-ready tables and trains the early-detection model.

In [ ]:
from pathlib import Path
import json
import random
import copy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import (
    pad_sequence,
    pack_padded_sequence,
    pad_packed_sequence,
)

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
    precision_recall_curve,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

PROJECT_ROOT = Path("..").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ML_DIR = PROCESSED_DIR / "ml"
LSTM_DIR = PROCESSED_DIR / "lstm"
LSTM_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = ML_DIR / "train_processed.csv"
VALIDATION_PATH = ML_DIR / "validation_processed.csv"
TEST_PATH = ML_DIR / "test_processed.csv"
FEATURE_NAMES_PATH = ML_DIR / "feature_names.json"
MANIFEST_PATH = ML_DIR / "sequence_manifest.csv"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

# 1) Loading ML Data

In [ ]:
with open(FEATURE_NAMES_PATH, "r", encoding="utf-8") as f:
    feature_names = json.load(f)

train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VALIDATION_PATH)
test_df = pd.read_csv(TEST_PATH)
sequence_manifest = pd.read_csv(MANIFEST_PATH)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)
print("Feature count:", len(feature_names))

# 2) Building Variable-Length Sequences

Each `sequence_id` is one student/module/presentation sequence.

In [ ]:
class SequenceDataset(Dataset):
    def __init__(
        self,
        X,
        y,
        prediction_mask,
        step_weights,
        lengths,
        sequence_ids,
        step_numbers,
    ):
        self.X = X
        self.y = y
        self.prediction_mask = prediction_mask
        self.step_weights = step_weights
        self.lengths = lengths
        self.sequence_ids = sequence_ids
        self.step_numbers = step_numbers

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return (
            self.X[idx],
            self.y[idx],
            self.prediction_mask[idx],
            self.step_weights[idx],
            self.lengths[idx],
            self.sequence_ids[idx],
            self.step_numbers[idx],
        )


def build_sequences(df, feature_names):
    sequences = []
    targets = []
    masks = []
    lengths = []
    sequence_ids = []
    step_lists = []

    for seq_id, group in df.groupby("sequence_id", sort=False):
        group = group.sort_values("time_step")

        sequences.append(
            torch.tensor(
                group[feature_names].to_numpy(dtype=np.float32)
            )
        )

        targets.append(
            torch.tensor(
                group["target"].to_numpy(dtype=np.float32)
            )
        )

        masks.append(
            torch.tensor(
                group["is_prediction_step"].to_numpy(dtype=np.float32)
            )
        )

        lengths.append(len(group))
        sequence_ids.append(int(seq_id))

        step_lists.append(
            torch.tensor(
                group["time_step"].to_numpy(dtype=np.int64)
            )
        )

    X_pad = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=0.0,
    )

    y_pad = pad_sequence(
        targets,
        batch_first=True,
        padding_value=0.0,
    )

    mask_pad = pad_sequence(
        masks,
        batch_first=True,
        padding_value=0.0,
    )

    step_pad = pad_sequence(
        step_lists,
        batch_first=True,
        padding_value=-1,
    )

    lengths = torch.tensor(lengths, dtype=torch.long)
    sequence_ids = torch.tensor(sequence_ids, dtype=torch.long)

    # Each sequence contributes equally across its valid prediction steps.
    valid_counts = mask_pad.sum(dim=1, keepdim=True).clamp_min(1.0)
    step_weights = mask_pad / valid_counts

    return (
        X_pad,
        y_pad,
        mask_pad,
        step_weights,
        lengths,
        sequence_ids,
        step_pad,
    )

train_tensors = build_sequences(train_df, feature_names)
validation_tensors = build_sequences(validation_df, feature_names)
test_tensors = build_sequences(test_df, feature_names)

In [ ]:
for name, tensors in [
    ("train", train_tensors),
    ("validation", validation_tensors),
    ("test", test_tensors),
]:
    _, _, _, _, lengths, sequence_ids, _ = tensors
    print(
        name,
        "sequences=", len(sequence_ids),
        "min_length=", int(lengths.min()),
        "max_length=", int(lengths.max()),
        "mean_length=", float(lengths.float().mean()),
    )

# 3) DataLoaders

In [ ]:
train_dataset = SequenceDataset(*train_tensors)
validation_dataset = SequenceDataset(*validation_tensors)
test_dataset = SequenceDataset(*test_tensors)

BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    num_workers=0,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

# 4) Class Imbalance

In [ ]:
train_sequence_targets = (
    train_df
    .groupby("sequence_id")["target"]
    .first()
)

positive_sequences = int((train_sequence_targets == 1).sum())
negative_sequences = int((train_sequence_targets == 0).sum())

if positive_sequences == 0:
    raise ValueError("No positive training sequences found.")

POS_WEIGHT = torch.tensor(
    [negative_sequences / positive_sequences],
    dtype=torch.float32,
    device=DEVICE,
)

print("Positive train sequences:", positive_sequences)
print("Negative train sequences:", negative_sequences)
print("pos_weight:", float(POS_WEIGHT.item()))

# 5) Causal LSTM Model

The LSTM is one-directional so the prediction at `t` uses the prefix up to `t`.

In [ ]:
class CausalLSTM(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_size=64,
        num_layers=2,
        dropout=0.20,
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False,
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, x, lengths):
        packed = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        packed_output, _ = self.lstm(packed)

        output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=x.size(1),
        )

        logits = self.classifier(
            self.dropout(output)
        ).squeeze(-1)

        return logits

model = CausalLSTM(
    input_size=len(feature_names),
    hidden_size=64,
    num_layers=2,
    dropout=0.20,
).to(DEVICE)

print(model)

# 6) Loss & Optimizer

In [ ]:
criterion = nn.BCEWithLogitsLoss(
    reduction="none",
    pos_weight=POS_WEIGHT,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

MAX_EPOCHS = 50
PATIENCE = 7
GRAD_CLIP = 1.0

# 7) Metrics & Prediction

In [ ]:
def masked_metrics(y_true, probabilities, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    probabilities = np.asarray(probabilities)
    predictions = (probabilities >= threshold).astype(int)

    return {
        "pr_auc": (
            average_precision_score(y_true, probabilities)
            if y_true.sum() > 0 else np.nan
        ),
        "roc_auc": (
            roc_auc_score(y_true, probabilities)
            if len(np.unique(y_true)) == 2 else np.nan
        ),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "brier": brier_score_loss(y_true, probabilities),
    }


def get_predictions(model, loader, device):
    model.eval()

    all_probabilities = []
    all_targets = []
    all_steps = []
    all_sequence_ids = []

    with torch.no_grad():
        for (
            X,
            y,
            prediction_mask,
            step_weights,
            lengths,
            sequence_ids,
            step_numbers,
        ) in loader:
            X = X.to(device)
            lengths = lengths.to(device)

            logits = model(X, lengths)
            probabilities = torch.sigmoid(logits).cpu()

            valid = prediction_mask > 0

            all_probabilities.append(probabilities[valid].numpy())
            all_targets.append(y[valid].numpy())

            expanded_sequence_ids = (
                sequence_ids
                .unsqueeze(1)
                .expand_as(step_numbers)
            )

            all_sequence_ids.append(
                expanded_sequence_ids[valid].numpy()
            )

            all_steps.append(step_numbers[valid].numpy())

    return (
        np.concatenate(all_probabilities),
        np.concatenate(all_targets),
        np.concatenate(all_sequence_ids),
        np.concatenate(all_steps),
    )

# 8) Training

In [ ]:
def train_one_epoch(model, loader):
    model.train()

    running_loss = 0.0

    for (
        X,
        y,
        prediction_mask,
        step_weights,
        lengths,
        sequence_ids,
        step_numbers,
    ) in loader:
        X = X.to(DEVICE)
        y = y.to(DEVICE)
        step_weights = step_weights.to(DEVICE)
        lengths = lengths.to(DEVICE)

        optimizer.zero_grad()

        logits = model(X, lengths)
        step_loss = criterion(logits, y)

        # Padding and Exam steps have zero weight.
        loss = (step_loss * step_weights).sum() / X.size(0)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        running_loss += float(loss.item())

    return running_loss / max(len(loader), 1)

In [ ]:
history = []
best_validation_pr_auc = -np.inf
best_state = None
patience_counter = 0

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader)

    validation_prob, validation_true, _, _ = get_predictions(
        model,
        validation_loader,
        DEVICE,
    )

    validation_metrics = masked_metrics(
        validation_true,
        validation_prob,
        threshold=0.5,
    )

    current_pr_auc = validation_metrics["pr_auc"]

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        **{f"validation_{key}": value for key, value in validation_metrics.items()},
    })

    print(
        f"Epoch {epoch:02d} | "
        f"loss={train_loss:.4f} | "
        f"validation_pr_auc={current_pr_auc:.4f} | "
        f"validation_roc_auc={validation_metrics['roc_auc']:.4f}"
    )

    if current_pr_auc > best_validation_pr_auc:
        best_validation_pr_auc = current_pr_auc
        best_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("Early stopping.")
        break

if best_state is None:
    raise RuntimeError("No best model state was captured.")

model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
history_df.to_csv(
    LSTM_DIR / "training_history.csv",
    index=False,
)

print("Best validation PR-AUC:", best_validation_pr_auc)

# 9) Choosing the Decision Threshold

The threshold is selected from Validation only.

In [ ]:
validation_prob, validation_true, validation_sequence_ids, validation_steps = get_predictions(
    model,
    validation_loader,
    DEVICE,
)

precision, recall, thresholds = precision_recall_curve(
    validation_true,
    validation_prob,
)

if len(thresholds) == 0:
    raise RuntimeError("Validation set did not provide thresholds.")

f1_curve = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-12)
)

best_idx = int(np.nanargmax(f1_curve))
threshold = float(thresholds[best_idx])

validation_metrics = masked_metrics(
    validation_true,
    validation_prob,
    threshold,
)

print("Decision threshold:", threshold)
print("Validation metrics:", validation_metrics)

with open(LSTM_DIR / "decision_threshold.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "threshold": threshold,
            "selection_metric": "validation_F1",
        },
        f,
        indent=2,
    )

# 10) Final Test Evaluation

In [ ]:
test_prob, test_true, test_sequence_ids, test_steps = get_predictions(
    model,
    test_loader,
    DEVICE,
)

test_metrics = masked_metrics(
    test_true,
    test_prob,
    threshold,
)

print("=== TEST METRICS ===")
for key, value in test_metrics.items():
    print(f"{key}: {value:.6f}")

with open(LSTM_DIR / "test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2)

## 10.1 Test Predictions by Timestep

In [ ]:
test_predictions = pd.DataFrame({
    "sequence_id": test_sequence_ids.astype(int),
    "time_step": test_steps.astype(int),
    "target": test_true.astype(int),
    "risk_probability": test_prob,
})

test_predictions["prediction"] = (
    test_predictions["risk_probability"] >= threshold
).astype(int)

test_predictions = test_predictions.merge(
    sequence_manifest[["sequence_id", "id_student"]],
    on="sequence_id",
    how="left",
    validate="many_to_one",
)

test_context = (
    test_df[[
        "sequence_id",
        "code_module",
        "code_presentation",
    ]]
    .drop_duplicates("sequence_id")
)

test_predictions = test_predictions.merge(
    test_context,
    on="sequence_id",
    how="left",
    validate="many_to_one",
)

test_predictions.to_csv(
    LSTM_DIR / "test_predictions_by_timestep.csv",
    index=False,
)

test_predictions.head(20)

# 11) Performance by Timestep

In [ ]:
per_timestep = []

for step in sorted(test_predictions["time_step"].unique()):
    part = test_predictions[
        test_predictions["time_step"] == step
    ]

    if part["target"].nunique() < 2:
        pr_auc = np.nan
        roc_auc = np.nan
    else:
        pr_auc = average_precision_score(
            part["target"],
            part["risk_probability"],
        )
        roc_auc = roc_auc_score(
            part["target"],
            part["risk_probability"],
        )

    pred = (part["risk_probability"] >= threshold).astype(int)

    per_timestep.append({
        "time_step": int(step),
        "n_checkpoints": len(part),
        "positive_ratio": float(part["target"].mean()),
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
        "precision": precision_score(part["target"], pred, zero_division=0),
        "recall": recall_score(part["target"], pred, zero_division=0),
        "f1": f1_score(part["target"], pred, zero_division=0),
    })

per_timestep_df = pd.DataFrame(per_timestep)
print(per_timestep_df)

per_timestep_df.to_csv(
    LSTM_DIR / "test_metrics_by_timestep.csv",
    index=False,
)

# 12) Early Detection Analysis

In [ ]:
positive_test_sequences = set(
    sequence_manifest[
        (sequence_manifest["split"] == "test")
        & (sequence_manifest["target"] == 1)
        & (sequence_manifest["has_prediction_step"] == 1)
    ]["sequence_id"].astype(int)
)

first_detection = []

for seq_id in sorted(positive_test_sequences):
    part = test_predictions[
        test_predictions["sequence_id"] == seq_id
    ].sort_values("time_step")

    hits = part[
        part["risk_probability"] >= threshold
    ]

    first_step = (
        int(hits["time_step"].iloc[0])
        if len(hits) > 0
        else np.nan
    )

    first_detection.append({
        "sequence_id": int(seq_id),
        "first_detected_step": first_step,
    })

first_detection_df = pd.DataFrame(first_detection)

n_positive = len(first_detection_df)
n_detected = int(
    first_detection_df["first_detected_step"].notna().sum()
)

overall_detection_rate = (
    n_detected / n_positive
    if n_positive > 0 else np.nan
)

median_detection_step = (
    first_detection_df["first_detected_step"].median()
)

print("Positive test sequences:", n_positive)
print("Detected:", n_detected)
print("Detection rate:", overall_detection_rate)
print("Median first detection step:", median_detection_step)

first_detection_df.to_csv(
    LSTM_DIR / "first_detection_by_sequence.csv",
    index=False,
)

In [ ]:
detection_by_step = []

for step in range(
    1,
    int(test_predictions["time_step"].max()) + 1,
):
    detected_by_step = (
        first_detection_df[
            first_detection_df["first_detected_step"].notna()
        ]["first_detected_step"]
        <= step
    ).sum()

    detection_by_step.append({
        "time_step": step,
        "positive_sequences": n_positive,
        "detected_by_this_step": int(detected_by_step),
        "detection_rate": (
            detected_by_step / n_positive
            if n_positive > 0 else np.nan
        ),
    })

detection_by_step_df = pd.DataFrame(detection_by_step)
print(detection_by_step_df)

detection_by_step_df.to_csv(
    LSTM_DIR / "detection_rate_by_timestep.csv",
    index=False,
)

# 13) Saving the LSTM Artifact

In [ ]:
MODEL_CONFIG = {
    "input_size": len(feature_names),
    "hidden_size": 64,
    "num_layers": 2,
    "dropout": 0.20,
    "bidirectional": False,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "seed": SEED,
    "prediction_steps": "non-Exam checkpoints only",
}

checkpoint = {
    "model_state_dict": model.state_dict(),
    "model_config": MODEL_CONFIG,
    "feature_names": feature_names,
    "threshold": threshold,
}

torch.save(
    checkpoint,
    LSTM_DIR / "causal_lstm.pt",
)

with open(LSTM_DIR / "model_config.json", "w", encoding="utf-8") as f:
    json.dump(MODEL_CONFIG, f, indent=2)

print("Saved model:", LSTM_DIR / "causal_lstm.pt")

# 14) Final LSTM Audit

In [ ]:
print("=== FINAL LSTM AUDIT ===")
print("Input features:", len(feature_names))
print("Best validation PR-AUC:", best_validation_pr_auc)
print("Decision threshold:", threshold)
print("Test PR-AUC:", test_metrics["pr_auc"])
print("Test ROC-AUC:", test_metrics["roc_auc"])
print("Test F1:", test_metrics["f1"])
print("Test Recall:", test_metrics["recall"])
print("First-detection rate:", overall_detection_rate)
print("Median first-detection step:", median_detection_step)

# ✅ What happened in this file?

We converted the processed tables into variable-length sequences, trained a one-directional LSTM, evaluated it on unseen students, and saved timestep-level predictions and early-detection results.